# MSC GeoMet - Canadian Precipitation Analysis (CaPA) example

CaPA provides gridded estimates of precipitation produced in near real-time. In other words, CaPA offers a “best-guess” of precipitation amounts in recent hours. Depending on the CaPA configuration, these estimates are available at any given location in Canada (RDPA), or in most of Canada, excluding the Canadian Arctic Archipelago (HRDPA/HREPA). All CaPA configurations provide precipitation estimates even in areas not well serviced by observation stations. CaPA analyses are generated by merging records from ground-based precipitation gauges, weather radar, and information from one of ECCC’s NWP systems (the NWP system varies by CaPA configuration). There are three different configurations of CaPA: the Regional Deterministic Precipitation Analysis (RDPA), and the High Resolution Deterministic and Ensemble Precipitation Analyses (HRDPA and HREPA). 

This tutorial will focus on using OGC Coverages API to query RDPA data. OGC Coverages API allows us to access archived RDPA data back to 2012.

## Requirements

First, we import all necessary Python libraries.  Some of these are standard Python libraries, while others are third-party libraries that must first be installed in your Python environment, namely [OWSLib](https://geopython.github.io/OWSLib/index.html "OWSLib Documentation"), which stands for the Open Geospatial Consortium (OGC) web service and which will be used for accessing the Coverages API.

Full technical documentation for the OGC Coverages API can be found here: https://eccc-msc.github.io/open-data/msc-geomet/ogc_api_en/

In [21]:
# data
import json
import numpy as np
import xarray as xr
from shapely.geometry import Point
import geopandas as gpd

# OGC Coverages API
from owslib.ogcapi.coverages import Coverages

# plotting
import hvplot.xarray

## Layers and Metadata

To see what RDPA layers MSC Geomet's OGC Coverages API has available, let's first connect and explore.  

In [3]:
cov = Coverages("https://api.weather.gc.ca/")
collections = cov.collections()

In [4]:
rdpa = [
    c for c in collections["collections"]
    if "rdpa" in c["id"].lower()
]

for c in rdpa:
    print(c["id"])
    print("  Title:", c.get("title", ""))
    print()

weather:rdpa:15km:24f
  Title: RDPA - 24 Hours Accumulation at 15 km Resolution

weather:rdpa:15km:6f
  Title: RDPA - 6 Hours Accumulation at 15 km Resolution

weather:rdpa:10km:24f
  Title: RDPA - 24 Hours Accumulation at 10 km Resolution

weather:rdpa:10km:6f
  Title: RDPA - 6 Hours Accumulation at 10 km Resolution

weather:rdpa:10km:24p
  Title: RDPA - 24 Hours Accumulation at 10 km Resolution (Preliminary)

weather:rdpa:10km:6p
  Title: RDPA - 6 Hours Accumulation at 10 km Resolution (Preliminary)



Now we know which RDPA layers are available! The first line printed for the layers is the collection ID we need for querying, and the title shows the long name. 

At the end of each ID is either an "f" or a "p". The "f" stands for final, while the "p" stands for preliminary.

## Querying RDPA as JSON

Let's query the RDPA 24 hour precipitation accumulation (final). First, we need to make the request for the RDPA layer in coverageJSON response.

In [5]:
# define date for coverage request
time = "2025-01-01T12Z"

In [6]:
response = cov.coverage(
    collection_id="weather:rdpa:10km:24f",
    datetime=time,
    f="json"
)

# CoverageJSON response
data = json.loads(response.getvalue().decode())

Using the domain of the data we queried, we can build xy coordinates.

In [7]:
x = np.linspace(
    data["domain"]["axes"]["x"]["start"],
    data["domain"]["axes"]["x"]["stop"],
    data["domain"]["axes"]["x"]["num"]
)

y = np.linspace(
    data["domain"]["axes"]["y"]["start"],
    data["domain"]["axes"]["y"]["stop"],
    data["domain"]["axes"]["y"]["num"]
)

Now we can extract the precipitation values. Note that "APCP" here stands for "accumulated precipitation / précipitations cumulées".

In [8]:
values = np.array(
    data["ranges"]["APCP"]["values"],
    dtype=np.float32
)

We can now map the precipitation values to the grid.

In [9]:
shape = tuple(data["ranges"]["APCP"]["shape"])
values = values.reshape(shape)

In [10]:
values

array([[8.0415001e+00, 5.7870002e+00, 4.1732502e+00, ..., 9.9990000e+03,
        9.9990000e+03, 9.9990000e+03],
       [7.0535002e+00, 6.5385003e+00, 5.5160003e+00, ..., 9.9990000e+03,
        9.9990000e+03, 9.9990000e+03],
       [5.2910004e+00, 5.2527504e+00, 5.1690001e+00, ..., 9.9990000e+03,
        9.9990000e+03, 9.9990000e+03],
       ...,
       [2.4147501e+00, 1.8767501e+00, 1.4457501e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [3.1012502e+00, 2.5977502e+00, 2.0485001e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [3.5020001e+00, 3.1395001e+00, 2.7085001e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00]], shape=(824, 935), dtype=float32)

### Change Data to xarray Dataset

In [11]:
ds = xr.Dataset(
    data_vars={
        "APCP": (
            ("y", "x"),
            values
        )
    },
    coords={
        "x": x,
        "y": y,
        "time": "2025-01-01T12Z"
    },
    attrs={
        "crs": data["domain"]["referencing"][0]["system"]["id"]
    }
)

In [12]:
ds

<xarray.Dataset> Size: 3MB
Dimensions:  (y: 824, x: 935)
Coordinates:
  * y        (y) float64 7kB 9.207e+05 9.107e+05 ... -7.309e+06 -7.319e+06
  * x        (x) float64 7kB -4.556e+06 -4.546e+06 ... 4.784e+06 4.794e+06
    time     <U14 56B '2025-01-01T12Z'
Data variables:
    APCP     (y, x) float32 3MB 8.042 5.787 4.173 3.044 2.22 ... 0.0 0.0 0.0 0.0
Attributes:
    crs:      http://www.opengis.net/def/crs/OGC/1.3/None

Now let's add metadata for the long name and units of our data from the JSON file to this dataset.

In [13]:
ds["APCP"].attrs["long_name"] = (
    data["parameters"]["APCP"]["description"]
)

ds["APCP"].attrs["units"] = (
    data["parameters"]["APCP"]["unit"]["symbol"]
)

In [14]:
print(ds["APCP"])

<xarray.DataArray 'APCP' (y: 824, x: 935)> Size: 3MB
array([[8.0415001e+00, 5.7870002e+00, 4.1732502e+00, ..., 9.9990000e+03,
        9.9990000e+03, 9.9990000e+03],
       [7.0535002e+00, 6.5385003e+00, 5.5160003e+00, ..., 9.9990000e+03,
        9.9990000e+03, 9.9990000e+03],
       [5.2910004e+00, 5.2527504e+00, 5.1690001e+00, ..., 9.9990000e+03,
        9.9990000e+03, 9.9990000e+03],
       ...,
       [2.4147501e+00, 1.8767501e+00, 1.4457501e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [3.1012502e+00, 2.5977502e+00, 2.0485001e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [3.5020001e+00, 3.1395001e+00, 2.7085001e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00]], shape=(824, 935), dtype=float32)
Coordinates:
  * y        (y) float64 7kB 9.207e+05 9.107e+05 ... -7.309e+06 -7.319e+06
  * x        (x) float64 7kB -4.556e+06 -4.546e+06 ... 4.784e+06 4.794e+06
    time     <U14 56B '2025-01-01T12Z'
Attributes:
    long_name: 

In [ ]:
## graph something with this?

### Save as a NetCDF

Saving requested data to a netCDF file is very easy with xarray.

In [40]:
# save to netcdf
ds.to_netcdf(path='APCP_test_data.nc')

### Save as a geoJSON

Saving the data to geoJSON is also simple, but takes a few more steps.

In [15]:
df = ds.to_dataframe()
df.head()

APCP            time
y             x                                     
920682.141166 -4.556441e+06  8.04150  2025-01-01T12Z
              -4.546431e+06  5.78700  2025-01-01T12Z
              -4.536420e+06  4.17325  2025-01-01T12Z
              -4.526409e+06  3.04425  2025-01-01T12Z
              -4.516399e+06  2.22025  2025-01-01T12Z

In [ ]:
df = df.reset_index()
df.head()

,y,x,APCP,time
0,920682.141166,-4.556441e+06,8.04150,2025-01-01T12Z
1,920682.141166,-4.546431e+06,5.78700,2025-01-01T12Z
2,920682.141166,-4.536420e+06,4.17325,2025-01-01T12Z
3,920682.141166,-4.526409e+06,3.04425,2025-01-01T12Z
4,920682.141166,-4.516399e+06,2.22025,2025-01-01T12Z


In [17]:
# bundle the latitudes and longitude in Point geometry tuples
geom = [Point(x,y) for x, y in zip(df['x'], df['y'])]

In [18]:
# transform the DataFrame into a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geom, crs='EPSG:4326')
gdf.head()

,y,x,APCP,time,geometry
0,920682.141166,-4.556441e+06,8.04150,2025-01-01T12Z,POINT (-4556441.40332 920682.14117)
1,920682.141166,-4.546431e+06,5.78700,2025-01-01T12Z,POINT (-4546430.69668 920682.14117)
2,920682.141166,-4.536420e+06,4.17325,2025-01-01T12Z,POINT (-4536419.99004 920682.14117)
3,920682.141166,-4.526409e+06,3.04425,2025-01-01T12Z,POINT (-4526409.2834 920682.14117)
4,920682.141166,-4.516399e+06,2.22025,2025-01-01T12Z,POINT (-4516398.57676 920682.14117)


In [19]:
# remove the unneeded latitude and longitude series
gdf = gdf.drop(['x', 'y'], axis=1)
gdf.head()

,APCP,time,geometry
0,8.04150,2025-01-01T12Z,POINT (-4556441.40332 920682.14117)
1,5.78700,2025-01-01T12Z,POINT (-4546430.69668 920682.14117)
2,4.17325,2025-01-01T12Z,POINT (-4536419.99004 920682.14117)
3,3.04425,2025-01-01T12Z,POINT (-4526409.2834 920682.14117)
4,2.22025,2025-01-01T12Z,POINT (-4516398.57676 920682.14117)


In [ ]:
# save to json
gdf.to_file('APCP_test_data.json', driver='GeoJSON')

### Plotting

!!! need to transform the CRS before plotting

In [23]:
ds.hvplot(width=800, height=700, cmap='viridis', cnorm='linear')

:Image   [y,x]   (Total precipitation [kg/(m^2)])

## Querying RDPA Data as GRIB2

If we want to query RDPA data over a certain area, we can define a bounding box.